# Why do we price options as if nobody expects a return?

**Out of Sample** — [read the article](https://arthurnoo.pythonanywhere.com/why-options-ignore-your-forecast)

This notebook reproduces every figure and every number in the article.

## Three ways to run it

| Where | What you need | Link |
| --- | --- | --- |
| **In your browser** | nothing at all | [Open in Colab](https://colab.research.google.com/github/Arthurnoo/out-of-sample/blob/main/notebooks/risk-neutral-pricing.ipynb) |
| **In VS Code** | VS Code installed | [Clone and open](vscode://vscode.git/clone?url=https://github.com/Arthurnoo/out-of-sample) |
| **Anywhere else** | Python 3.10+ | you already have the file |

The next cell installs whatever is missing, so in every case you can simply
**Run All** and watch. The first run downloads prices and takes a minute or two.

## How to read it

Each section matches a section of the article. Run a cell, look at what it
prints, then read the note underneath explaining what to make of it.

**Section 0 holds every parameter you might want to change.** Everything after
it is written so that changing those values changes the whole notebook. That is
the point: the numbers below are mine, and the interesting question is what
happens to them when you move something.

Your numbers may differ slightly from the article if you run this later. The
data keeps growing, which is itself worth noticing.

## If you find a mistake

Tell me. There is a form at the bottom of the article, and a wrong number
corrected in public is worth more than a right one nobody checked.

In [ ]:
# Installs anything missing, then does nothing on later runs.
# This is what makes "Run All" work in Colab, in VS Code, or on a bare machine.
import importlib, subprocess, sys

for package in ['yfinance', 'pandas', 'numpy', 'matplotlib', 'scipy']:
    try:
        importlib.import_module(package)
    except ImportError:
        print(f'installing {package} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package], check=True)

print('ready')

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import datetime as dt
from math import log, sqrt, exp, erf

pd.set_option('display.width', 120)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '#e4e9ee', 'axes.facecolor': 'white'})

## 0. The knobs

Everything worth changing is in the next cell.

| Knob | What it does | Try |
| --- | --- | --- |
| `UP`, `DOWN` | the two outcomes in the one-period example | `1.5`, `0.5` for a wilder stock |
| `SIGMA` | the volatility used in the hedging simulation | `0.40`, to see the error grow |
| `N_PATHS` | simulated paths; more is slower but smoother | `1000` to go faster |
| `REBALANCES` | how often the hedge is adjusted | add `1000` |
| `UNDERLYING` | the ticker whose option chain we pull | `QQQ`, `AAPL`, `TLT` |
| `TARGET_DAYS` | how far out the expiry should be | `(80, 150)` for longer options |
| `RATE` | the risk-free rate assumed | match the current T-bill |
| `MIN_VOLUME` | how liquid a strike must be to be used | raise it and the fit improves |

`SIGMA` and `REBALANCES` drive the second section. `UNDERLYING` and
`MIN_VOLUME` drive the last two, and raising `MIN_VOLUME` is the single change
most likely to make the density extraction behave.

In [ ]:
# ─────────────────────────────────────────────────────────────────────
#  PARAMETERS — this is the only cell you need to edit
# ─────────────────────────────────────────────────────────────────────

# Section 1, the one-period example
S0_TOY, UP, DOWN, STRIKE_TOY, RATE_TOY = 100.0, 1.10, 0.90, 100.0, 0.0

# Section 2, the hedging simulation
S0, STRIKE, MATURITY, RATE, SIGMA = 100.0, 100.0, 1.0, 0.02, 0.20
N_PATHS    = 3000                       # simulated price paths
N_STEPS    = 253                        # daily steps over one year
REBALANCES = [1, 4, 12, 52, 126, 252]   # hedging frequencies to compare
DRIFTS     = [-0.10, 0.0, 0.08, 0.20]   # real-world expected returns to test

# Sections 3 and 4, the real option chain
UNDERLYING  = 'SPY'
TARGET_DAYS = (25, 70)    # pick an expiry this many days out
MIN_VOLUME  = 10          # ignore strikes that traded less than this today

## 1. The smallest possible example

One period, two outcomes. A stock at 100 goes to 110 or to 90. A call struck at
100 pays 10 or 0. Rates are zero.

Instead of guessing the probability, we **build** the option out of stock and
cash. Two equations, two unknowns.

In [ ]:
def replicate_one_step(S0, up, down, K, r):
    Su, Sd = S0 * up, S0 * down
    Cu, Cd = max(Su - K, 0.0), max(Sd - K, 0.0)

    # solve:  delta*Su + cash*(1+r) = Cu
    #         delta*Sd + cash*(1+r) = Cd
    delta = (Cu - Cd) / (Su - Sd)
    cash = (Cd - delta * Sd) / (1 + r)
    price = delta * S0 + cash                 # the cost of the ingredients

    q = ((1 + r) - down) / (up - down)        # the pseudo-probability
    return dict(delta=delta, cash=cash, price=price, q=q, Cu=Cu, Cd=Cd)

rep = replicate_one_step(S0_TOY, UP, DOWN, STRIKE_TOY, RATE_TOY)
print(f"hold {rep['delta']:.2f} share, borrow {-rep['cash']:.0f}")
print(f"cost today = {rep['price']:.2f}")
print(f"pseudo-probability q = {rep['q']:.2f}")
print(f"same answer via q: {(rep['q']*rep['Cu'] + (1-rep['q'])*rep['Cd']):.2f}")

Now vary the thing everyone argues about: the real probability that the stock
goes up. Watch the expected payoff move by a factor of nineteen while the price
does not move at all.

In [ ]:
rows = []
for p_real in [0.05, 0.25, 0.50, 0.75, 0.95]:
    rows.append({'your belief it rises': p_real,
                 'expected payoff': p_real * rep['Cu'] + (1 - p_real) * rep['Cd'],
                 'replication price': rep['price']})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:20.2f}'))

`q` is not anybody's belief. It is defined by one property: under `q`, the stock
itself is expected to earn the risk-free rate. That is the whole content of the
phrase "risk-neutral measure".

## 2. Black-Scholes, and what breaks in practice

Same idea in continuous time. Note that `mu`, the expected return, appears
nowhere in these functions.

In [ ]:
def N(x):
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))

def bs_call(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return max(S - K, 0.0)
    d1 = (log(S / K) + (r + sigma * sigma / 2) * T) / (sigma * sqrt(T))
    return S * N(d1) - K * exp(-r * T) * N(d1 - sigma * sqrt(T))

def bs_delta(S, K, T, r, sigma):
    if T <= 0 or sigma <= 0:
        return 1.0 if S > K else 0.0
    d1 = (log(S / K) + (r + sigma * sigma / 2) * T) / (sigma * sqrt(T))
    return N(d1)

K, T, r, sigma = STRIKE, MATURITY, RATE, SIGMA
premium = bs_call(S0, K, T, r, sigma)
print(f"one-year call, strike 100, vol 20%, rate 2%  ->  {premium:.3f}")

### Does the price really not depend on the drift?

Simulate the stock under several real-world drifts, sell the call at the
Black-Scholes price, hedge it daily, and look at what is left at expiry. If the
theory holds, the leftover is centred on zero regardless of the drift.

In [ ]:
def gbm_paths(S0, mu, sigma, T, n_steps, n_paths, seed=0):
    rng = np.random.default_rng(seed)
    dt_ = T / (n_steps - 1)
    z = rng.standard_normal((n_paths, n_steps - 1))
    inc = (mu - sigma**2 / 2) * dt_ + sigma * sqrt(dt_) * z
    out = np.empty((n_paths, n_steps)); out[:, 0] = S0
    out[:, 1:] = S0 * np.exp(np.cumsum(inc, axis=1))
    return out

def delta_hedge(paths, K, T, r, sigma, n_rebal):
    """Sell the call, hedge it n_rebal times, return the leftover at expiry."""
    n_paths, n_steps = paths.shape
    step = max(1, n_steps // n_rebal)
    dt_ = T / (n_steps - 1)
    err = np.empty(n_paths)
    for i in range(n_paths):
        S = paths[i]
        cash = bs_call(S[0], K, T, r, sigma)        # collect the premium
        d = bs_delta(S[0], K, T, r, sigma)
        cash -= d * S[0]                             # buy the hedge
        for t in range(step, n_steps, step):
            tau = max(T - t * dt_, 1e-9)
            cash *= exp(r * dt_ * step)
            new_d = bs_delta(S[t], K, tau, r, sigma)
            cash -= (new_d - d) * S[t]               # adjust
            d = new_d
        err[i] = cash + d * S[-1] - max(S[-1] - K, 0.0)
    return err

rows = []
for mu in DRIFTS:
    paths = gbm_paths(S0, mu, sigma, T, N_STEPS, N_PATHS, seed=1)
    e = delta_hedge(paths, K, T, r, sigma, n_rebal=252)
    rows.append({'true expected return': mu, 'mean hedging error': e.mean(),
                 'std of error': e.std()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:20.4f}'))

The drift does not matter. Your forecast really is irrelevant to the price.

### But continuous hedging is a fiction

Now hedge less often and watch the error grow.

In [ ]:
paths = gbm_paths(S0, 0.08, sigma, T, N_STEPS, N_PATHS, seed=2)
rows = []
for n in REBALANCES:
    e = delta_hedge(paths, K, T, r, sigma, n_rebal=n)
    rows.append({'rebalances per year': n, 'std of error': e.std(),
                 'as % of premium': e.std() / premium * 100})
hedge = pd.DataFrame(rows)
print(hedge.to_string(index=False, float_format=lambda v: f'{v:19.3f}'))

fig, ax = plt.subplots(figsize=(7, 3.4))
ax.plot(hedge['rebalances per year'], hedge['std of error'], marker='o', color='#2d6fa8', label='observed')
ref = hedge['std of error'].iloc[-1] * np.sqrt(252 / hedge['rebalances per year'])
ax.plot(hedge['rebalances per year'], ref, ls='--', color='#8b98a6', label=r'theory $1/\sqrt{n}$')
ax.set_xscale('log'); ax.set_yscale('log'); ax.legend()
ax.set_xlabel('rebalances per year'); ax.set_ylabel('std. dev. of hedging error')
plt.show()

Hedge weekly, which is already diligent, and the typical error is still about a
tenth of the premium. The error falls like one over the square root of the
number of adjustments (Boyle and Emanuel, 1980), which is the straight line on
the log-log plot.

**The arbitrage price is exact only in a limit nobody reaches.** Everything that
happens in a real options business happens in that gap.

## 3. What the market actually thinks

If Black-Scholes were right, every option on the same underlying with the same
expiry would imply one volatility. Let us check.

We back the volatility out of each quoted price by bisection: a call price rises
with volatility, so a simple bisection always converges, unlike Newton near the
boundaries.

In [ ]:
def implied_vol(price, S, K, T, r, lo=1e-4, hi=4.0):
    if price <= max(S - K * exp(-r * T), 0.0) or price <= 0:
        return np.nan
    for _ in range(100):
        mid = (lo + hi) / 2
        if bs_call(S, K, T, r, mid) < price:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

tk = yf.Ticker(UNDERLYING)
S = float(tk.history(period='1d')['Close'].iloc[-1])
today = dt.date.today()
lo_d, hi_d = TARGET_DAYS
candidates = [e for e in tk.options
              if lo_d <= (dt.date.fromisoformat(e) - today).days <= hi_d]
expiry = candidates[len(candidates) // 2]
TT = (dt.date.fromisoformat(expiry) - today).days / 365.0
RR = RATE        # a flat assumption; bootstrapping the curve would be better

ch = tk.option_chain(expiry).calls.copy()
ch['mid'] = np.where((ch.bid > 0) & (ch.ask > 0), (ch.bid + ch.ask) / 2, ch.lastPrice)
sel = ch[(ch.strike > S * 0.78) & (ch.strike < S * 1.18) &
         (ch['mid'] > 0.05) & (ch.volume.fillna(0) > 0)].copy()
sel['iv'] = [implied_vol(p, S, k, TT, RR) for p, k in zip(sel['mid'], sel['strike'])]
sel['moneyness'] = sel.strike / S
smile = sel.dropna(subset=['iv']).sort_values('strike')

print(f"{UNDERLYING} at {S:.2f}, expiry {expiry} ({TT*365:.0f} days), {len(smile)} usable calls")
atm = float(smile.iloc[(smile.moneyness - 1).abs().argmin()]['iv'])
print(f"at the money: {atm*100:.1f}%")
print(f"at 92% of spot: {smile[smile.moneyness < 0.92].iv.mean()*100:.1f}%")
print(f"at 106% of spot: {smile[smile.moneyness > 1.06].iv.mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(smile.moneyness * 100, smile.iv * 100, marker='o', ms=3, color='#2d6fa8')
ax.axhline(atm * 100, ls='--', color='#8b98a6', label=f'at the money {atm*100:.1f}%')
ax.set_xlabel('strike as % of spot'); ax.set_ylabel('implied volatility (%)'); ax.legend()
ax.set_title('If Black-Scholes were right, this line would be flat', loc='left')
plt.show()

## 4. Reading the market's own distribution, and failing

Breeden and Litzenberger (1978): the risk-neutral density is the **second
derivative** of the call price with respect to strike.

A very tight call spread around K pays 1 if and only if the stock finishes at K,
so its price is the discounted probability of that event. Squeeze the spread and
you get the density.

It is beautiful and it does not survive contact with real quotes. Here is why,
which is more useful than the result would have been.

### First: are the quotes even convex?

No-arbitrage requires call prices to be a convex function of strike. If they are
not, a butterfly spread would have a negative price.

In [ ]:
raw = smile.sort_values('strike')
d2_raw = np.diff(raw['mid'].to_numpy(), 2)
print(f"convexity violations in the quotes: {(d2_raw < 0).sum()} out of {len(d2_raw)} strike triplets")
print("Stale prices, zero bid-ask on thin strikes, quotes from different moments.")

### Second: what that does to the density

In [ ]:
g0 = np.linspace(raw.strike.min(), raw.strike.max(), 300)
iv0 = np.interp(g0, raw.strike.values, raw.iv.values)
C0 = np.array([bs_call(S, k, TT, RR, v) for k, v in zip(g0, iv0)])

cdf0 = 1 + np.exp(RR * TT) * np.gradient(C0, g0)        # Q(S_T <= K)
q0 = np.exp(RR * TT) * np.gradient(np.gradient(C0, g0), g0)
trap = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz

print(f"total probability mass : {trap(np.clip(q0, 0, None), g0):.3f}   (must be 1)")
print(f"cumulative range       : {cdf0.min():.3f} to {cdf0.max():.3f}   (must be within [0,1])")
print(f"monotone increasing    : {bool(np.all(np.diff(cdf0) >= -1e-9))}")
print("\nNegative probabilities are not a numerical annoyance.")
print("They mean the input violated the arbitrage condition the method assumes.")

### Third: fitting a smooth smile fixes the symptom

Keep only strikes that actually traded, fit implied volatility with a quadratic
in log-moneyness, and re-price from the fit. The call curve is now twice
differentiable by construction.

In [ ]:
def fit_smile(strikes, ivs, S, T, r, deg=2):
    F = S * exp(r * T)
    k = np.log(np.asarray(strikes, float) / F)
    v = np.asarray(ivs, float)
    ok = np.isfinite(k) & np.isfinite(v) & (v > 0)
    poly = np.poly1d(np.polyfit(k[ok], v[ok], deg))
    rmse = float(np.sqrt(np.mean((poly(k[ok]) - v[ok]) ** 2)))
    return (lambda K: np.clip(poly(np.log(np.asarray(K, float) / F)), 1e-3, 5.0)), rmse, F

liq = smile[smile.volume >= MIN_VOLUME]
iv_fit, rmse, F = fit_smile(liq.strike.values, liq.iv.values, S, TT, RR)
print(f"{len(liq)} liquid calls, fit error {rmse*100:.2f} vol points")

g1 = np.linspace(S * 0.80, S * 1.20, 800)
C1 = np.array([bs_call(S, k, TT, RR, float(iv_fit(k))) for k in g1])
q1 = np.clip(np.exp(RR * TT) * np.gradient(np.gradient(C1, g1), g1), 0, None)
cdf1 = np.clip(1 + np.exp(RR * TT) * np.gradient(C1, g1), 0, 1)
q1n = q1 / trap(q1, g1)

print(f"density mass           : {trap(q1, g1):.3f}")
print(f"all non-negative       : {bool(np.all(q1 >= 0))}")
print(f"cumulative monotone    : {bool(np.all(np.diff(cdf1) >= -1e-9))}")
print(f"\nE[S_T] under Q        : {trap(g1*q1n, g1):.2f}")
print(f"forward price          : {F:.2f}")
print(f"gap                    : {(trap(g1*q1n, g1)/F - 1)*100:+.2f}%   <- this is the test it fails")

### Fourth: why it still fails

The forward test is the one that matters, because under the risk-neutral measure
the expected value of the stock **must** equal the forward by construction.

The reason is the wings. Outside the quoted strikes, a quadratic extrapolates
absurdly.

In [ ]:
rows = []
for frac in [0.5, 0.7, 0.85, 1.0, 1.15, 1.4, 2.0]:
    inside = raw.strike.min() <= S * frac <= raw.strike.max()
    rows.append({'% of spot': frac * 100, 'fitted vol %': float(iv_fit(S * frac)) * 100,
                 'status': 'observed' if inside else 'extrapolated'})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: f'{v:13.1f}'))

print("\nWiden the grid and the probability mass grows out of nothing:")
for lo, hi in [(0.80, 1.20), (0.60, 1.50), (0.40, 2.00)]:
    gg = np.linspace(S*lo, S*hi, 1500)
    CC = np.array([bs_call(S, k, TT, RR, float(iv_fit(k))) for k in gg])
    qq = np.clip(np.exp(RR*TT) * np.gradient(np.gradient(CC, gg), gg), 0, None)
    print(f"  grid {lo:.2f}S to {hi:.2f}S  ->  mass {trap(qq, gg):.3f}")

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 3.3), sharey=True)
a.plot(g0 / S * 100, cdf0, color='#c83c50'); a.axhline(0, color='k', lw=1); a.axhline(1, color='k', lw=1, ls=':')
a.set_title('from the quotes as they come', loc='left', fontsize=10); a.set_ylabel(r'$Q(S_T \leq K)$')
b.plot(g1 / S * 100, cdf1, color='#0e8f5f'); b.axhline(0, color='k', lw=1); b.axhline(1, color='k', lw=1, ls=':')
b.set_title('after fitting a smooth smile', loc='left', fontsize=10)
for ax_ in (a, b): ax_.set_xlabel('strike, % of spot')
plt.show()

The extra mass is not information about the market. It is invented by my own
extrapolation.

The real fix is to parametrise the smile in a form that cannot produce arbitrage
anywhere, including in the wings. That is what SVI and its arbitrage-free
variants are for (Gatheral and Jacquier, 2014).

## 5. Things worth trying yourself

1. **Run this during market hours.** How many convexity violations remain? My
   guess is far fewer, but not zero. Check whether the forward test passes then.
2. **Use puts instead of calls** for the strikes below spot. Out-of-the-money
   options are the liquid ones on each side, and mixing them via put-call parity
   gives a cleaner smile than using calls everywhere.
3. **Replace the quadratic fit with a cubic, then a quintic.** Watch the wings
   get worse, not better. More flexibility is the wrong answer here; the right
   answer is a form with the no-arbitrage conditions built in.
4. **Change the hedging simulation** so the stock has jumps or stochastic
   volatility. The hedging error stops being centred on zero, and you have
   discovered why selling options is not free money.

Found a mistake? There is a form at the bottom of the article. I would rather be
corrected in public than be quietly wrong.